> **Riverside's 70B problem:** You are the Platform Engineer at Riverside Publishing.
> For months your team has been fine-tuning a 7B model on editorial notes — it runs
> on a single A100 80 GB with room to spare. Last week, leadership approved a grant to
> train a _70B_ model on the full 7-novel catalog. IT provisioned exactly 4 × A100 80 GB GPUs.
>
> Your CTO wants an answer by Monday morning: "Can we train on what we have, or do we
> need to rent a 200-GPU cloud cluster?" Your single-GPU training script OOMs instantly
> at 70B — you know that much. But _why_? 140 GB for parameters alone at bf16; add
> gradients and Adam optimizer states and you need ~840 GB. You have 320 GB across four GPUs.
>
> Is this a memory problem? A code architecture problem? A hardware cost problem? All three?
> Three hours until Monday's meeting. This notebook is your analysis.


# Distributed Training: Scaling from One GPU to Many

| Part | Concept               | Why you need this NOW                                                        |
| ---- | --------------------- | ---------------------------------------------------------------------------- |
| 1    | Data Parallel (DDP)   | First instinct: put all 4 GPUs to work in parallel — does it even fit?       |
| 2    | FSDP                  | DDP keeps a full 840 GB copy on every GPU — need to shard or you never start |
| 3    | Tensor Parallelism    | One FFN weight matrix in 70B is 2 GB alone — wider than one GPU can hold     |
| 4    | Pipeline Parallelism  | 80 layers across 4 GPUs — how do you avoid 3 GPUs idling while 1 runs?       |
| 5    | Parallelism Selection | You need exactly ONE concrete strategy to present Monday morning             |
| 6    | Toy → Real Bridge     | Verify: does your derivation match Meta's actual LLaMA-2-70B recipe?         |

---

> **Prerequisites:** Ch2 Mixed Precision (memory math, bf16, LoRA)  
> **Running example:** A small Transformer model simulated across CPU process groups when GPU is unavailable


In [ ]:
import subprocess, sys

# Install torch/numpy/matplotlib only if missing, keeping the notebook runnable from a bare environment
for pkg in ["torch", "numpy", "matplotlib"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp
import numpy as np
import matplotlib.pyplot as plt
import os

# Detect real GPUs available to PyTorch; fall back to CPU process groups when none exist
HAS_GPU = torch.cuda.is_available()
N_GPUS = torch.cuda.device_count() if HAS_GPU else 0
DEVICE = torch.device("cuda" if HAS_GPU else "cpu")

print(f"GPU available: {HAS_GPU}")
print(f"Number of GPUs: {N_GPUS}")
# Report whether the multi-GPU demo will run on real hardware or CPU-simulated process groups
if N_GPUS >= 2:
    print("\u2713 Multi-GPU demo will run on actual GPUs")
else:
    print(
        "\u2192 Multi-GPU sections use CPU process groups (gradient math identical, timing not representative)"
    )
    print(
        "  Concepts and measurements are shown correctly \u2014 only timing differs from GPU hardware"
    )

print()
print("Riverside constraint: 4 A100 80GB GPUs for 70B model fine-tuning")
print("This notebook simulates the distributed training patterns")

---

## Part 1 — Data Parallel (DDP): Use All GPUs Simultaneously

Your single-GPU training loop does this: `loss.backward()` → `optimizer.step()`.
One GPU computes gradients for one mini-batch and updates weights. The other three
GPUs sit completely idle. That is free capacity doing nothing.

**DDP's insight:** every GPU can process a _different_ mini-batch simultaneously.
After each backward pass, they reconcile via an **all-reduce** operation — averaging
all GPUs' gradients into one shared signal — then every GPU applies the same update.
Weights stay in sync; throughput scales with GPU count.

**DDP** replicates the full model on every GPU. Each GPU processes a different mini-batch. After backward, gradients are **all-reduced** (averaged across GPUs) before the optimizer step.

The result: every GPU has identical weights after each step — as if we trained with a global batch size of `local_batch × n_gpus`.

#### #### Predict first

After DDP backward, are the gradients on GPU 0 and GPU 1:

1. **(a) Identical** — all-reduce averages them, so both GPUs have the same gradient
2. **(b) Summed (twice the magnitude)** — all-reduce adds them together
3. **(c) Each GPU keeps its own** — no communication happens in DDP backward

Which is correct?


> **Intuition:** Think of 4 engineers each reading a different section of Riverside's novel corpus. Each independently computes how to adjust the editing model (local gradient). Before anyone writes changes, they compare notes and use the average of all four signals. Every engineer now updates from the same averaged gradient — the model stays perfectly in sync across all 4 GPUs. This is the DDP all-reduce: local compute, then average, then update.


## Communication Vocabulary: From One Step to a Process Group

A single GPU step has one owner: forward, backward, then update. Distributed training keeps that step but gives each process a communication identity:

- **Rank:** a process's integer ID inside a distributed job. Rank 0 is the conventional coordinator, not a more powerful GPU.
- **Process group:** the set of ranks allowed to communicate. A job can have a global group plus smaller data-, tensor-, or pipeline-parallel groups.
- **Collective:** an operation every rank in a group enters, such as broadcast, all-gather, reduce-scatter, or all-reduce. One missing rank can stall the group.
- **All-reduce:** reduce values from all ranks, then distribute the result back to all ranks. The primitive usually computes a sum; DDP's scaling gives global-batch gradient semantics.
- **Replica:** a complete model copy. DDP keeps one replica per rank.
- **Shard:** one partition of a tensor or state. FSDP stores shards instead of persistent full replicas.
- **Bucket:** a contiguous buffer containing several gradients, large enough for efficient communication.
- **Overlap:** launch a bucket's collective while backward computation continues on earlier layers, hiding some communication time.

The important transition is not "one GPU becomes four GPUs." It is "one local gradient becomes a group agreement before any replica updates."

```mermaid
sequenceDiagram
    participant R0 as Rank 0 / GPU 0
    participant PG as Process group
    participant R1 as Rank 1 / GPU 1
    R0->>R0: Forward on local batch A
    R1->>R1: Forward on local batch B
    R0->>R0: Backward; bucket becomes ready
    R1->>R1: Backward; bucket becomes ready
    R0->>PG: Enter bucket all-reduce
    R1->>PG: Enter bucket all-reduce
    PG-->>R0: Same reduced gradient bucket
    PG-->>R1: Same reduced gradient bucket
    R0->>R0: Optimizer step
    R1->>R1: Optimizer step
```

**Image setup:** after making the synchronization-cost prediction, inspect `images/ddp-gradient-allreduce.png`. Track one local gradient into the ring and verify that every GPU exits with the same reduced value. The circular arrows are communication, not model movement.

###  Predict First: DDP Synchronisation Cost

Before running: each GPU has its own gradient. DDP must synchronise them. When does the synchronisation happen?

**A)** Before each forward pass — gradients from the last step guide the next  
**B)** After each backward pass — all-reduce happens while gradients are still fresh  
**C)** Only at the end of training — one big sync at checkpoint time

_Your prediction:_ \_\_\_

↓ Run the next cell to see the answer.


![DDP ring all-reduce: 4 GPU boxes each with local gradient tensors; circular teal arrow averages gradients so every GPU ends with identical weights](images/ddp-gradient-allreduce.png)


### Guided Reading: Why a Ring Instead of a Central Reducer?

Read the DDP image in three passes:

1. **Before the arrows:** each rank has a full model replica but a different local gradient.
2. **Along the arrows:** gradient chunks circulate; partial sums accumulate without one central GPU receiving everything.
3. **After the arrows:** each rank has the same reduced gradient and can update its own replica identically.

```mermaid
flowchart LR
    R0[Rank 0] -->|chunk| R1[Rank 1]
    R1 -->|chunk| R2[Rank 2]
    R2 -->|chunk| R3[Rank 3]
    R3 -->|chunk| R0
```

A ring all-reduce is two logical passes: **reduce-scatter** leaves each rank responsible for one reduced chunk, then **all-gather** circulates those chunks so every rank reconstructs the full result. With $N$ ranks and a gradient payload of size $S$, each rank sends approximately

$$\frac{2(N-1)}{N}S$$

bytes. For Riverside, bf16 gradients for 70B parameters are $70\text{B} \times 2 = 140$ GB. At $N=4$, each rank sends about $1.5 \times 140 = 210$ GB per step. A naive parameter server would make one server receive $3 \times 140 = 420$ GB and send another 420 GB, creating an 840 GB hotspot.

Buckets make this less stark in practice: backward produces later-layer gradients first, so DDP can all-reduce ready buckets while earlier layers are still computing. Only the communication left after backward is **exposed** in step time.

In [ ]:
#  Part 1: DDP gradient synchronization
# Simulate DDP's all-reduce on CPU (identical math to GPU DDP)


class ToyTransformer(nn.Module):
    """Small transformer-like model for demonstration."""

    def __init__(self, d=64, n_heads=4, n_layers=3):
        super().__init__()
        self.embed = nn.Embedding(100, d)
        # One TransformerEncoderLayer per depth level, stacked as a ModuleList
        self.layers = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(d, n_heads, batch_first=True)
                for _ in range(n_layers)
            ]
        )
        self.head = nn.Linear(d, 100)

    def forward(self, x):
        h = self.embed(x)
        # Run the input through every encoder layer in sequence
        for layer in self.layers:
            h = layer(h)
        return self.head(h)


torch.manual_seed(42)
model = ToyTransformer()
n_params = sum(p.numel() for p in model.parameters())
print(f"ToyTransformer: {n_params:,} parameters (simulates DDP mechanics)")
print()

# Simulate 2 GPUs with different batches
torch.manual_seed(0)
batch_gpu0 = torch.randint(0, 100, (4, 16))  # GPU 0's mini-batch
batch_gpu1 = torch.randint(0, 100, (4, 16))  # GPU 1's mini-batch (different)

# Forward + backward on each "GPU" (separate model copies)
# Both GPU replicas start from identical weights before their independent forward/backward passes
model0 = ToyTransformer()
model0.load_state_dict(model.state_dict())
model1 = ToyTransformer()
model1.load_state_dict(model.state_dict())
crit = nn.CrossEntropyLoss()

# GPU 0 forward/backward
out0 = model0(batch_gpu0)
labels0 = torch.randint(0, 100, (4, 16))
loss0 = crit(out0.view(-1, 100), labels0.view(-1))
loss0.backward()
grad0 = model0.embed.weight.grad.clone()

# GPU 1 forward/backward
out1 = model1(batch_gpu1)
labels1 = torch.randint(0, 100, (4, 16))
loss1 = crit(out1.view(-1, 100), labels1.view(-1))
loss1.backward()
grad1 = model1.embed.weight.grad.clone()

# DDP all-reduce: average gradients
ddp_grad = (grad0 + grad1) / 2

print("Before DDP all-reduce:")
print(f"  GPU 0 grad norm: {grad0.norm():.4f}")
print(f"  GPU 1 grad norm: {grad1.norm():.4f}")
print(f"  Are they identical? {torch.allclose(grad0, grad1)}")
print()
print("After DDP all-reduce (averaged):")
print(f"  Averaged grad norm: {ddp_grad.norm():.4f}")
print(f"  GPU 0 would see: {ddp_grad.norm():.4f}")
print(f"  GPU 1 would see: {ddp_grad.norm():.4f}  (identical!)")
print()
print("Prediction check: answer (a) \u2014 all-reduce AVERAGES gradients.")
print("Both GPUs update weights by the SAME gradient \u2192 weights stay in sync.")
print()
# Quantify how different the two GPUs' gradients were before averaging
diff = (grad0 - grad1).abs().mean().item()
print(f"Before sync: mean absolute gradient difference = {diff:.4f}")
print(f"After sync: difference = 0.0000  (by construction of all-reduce)")

In [ ]:
#  Part 1: DDP memory requirements
print("DDP memory requirement analysis for Riverside's 70B model:")
print()

model_params_b = 70  # billion parameters
bytes_per_param_bf16 = 2
# Convert billions of parameters to GB at 2 bytes/param (bf16)
params_gb = model_params_b * 1e9 * bytes_per_param_bf16 / 1e9
# DDP gradients are bf16, same footprint as the parameters
grads_gb  = params_gb
opt_gb    = model_params_b * 1e9 * 4 * 2 / 1e9  # fp32 Adam states

# DDP keeps a full copy of params + grads + optimizer state on every GPU
total_per_gpu_ddp = params_gb + grads_gb + opt_gb
a100_vram = 80  # GB

print(f"  70B model at bf16:             {params_gb:.0f} GB")
print(f"  Gradients (bf16):              {grads_gb:.0f} GB")
print(f"  Optimizer states (fp32 Adam):  {opt_gb:.0f} GB")
print(f"  Total per GPU with DDP:        {total_per_gpu_ddp:.0f} GB")
print(f"  A100 VRAM:                     {a100_vram} GB")
print()
print(f"  DDP status: {'\u2713 fits' if total_per_gpu_ddp <= a100_vram else '\u2717 OOM \u2014 needs FSDP or more VRAM'}")
print()
# Flag that a sharding strategy is required when the model doesn't fit even at bf16
if total_per_gpu_ddp > a100_vram:
    print(f"  \u2192 DDP cannot train 70B even with full bf16. Every GPU needs {total_per_gpu_ddp:.0f} GB.")
    print(f"    Need FSDP to shard parameters, gradients, and optimizer states across GPUs.")

#### What just happened — and what's missing

We confirmed the DDP all-reduce result: GPU 0 and GPU 1 end up with **identical
averaged** gradients — not different, not summed. The answer was **(a)**. Both GPUs
update weights from the same gradient signal, which is why DDP keeps all replicas
in perfect sync after every step.

But the memory cell delivers the harder verdict: DDP requires the **full model on
every GPU** at all times. For Riverside's 70B at bf16 with Adam, that is 840 GB per
GPU — ten times the 80 GB available. DDP cannot even load the model, let alone train.

→ **The complaint that forces Part 2:** we need a strategy that does not keep a full
copy of everything on every GPU. We need to _shard_ across the 4 GPUs instead.


In [ ]:
# #### Your turn — DDP memory for a 7B model
#
# The DDP analysis above was for 70B parameters.
# # CHANGE `model_params_b_yt` to 7 (the smaller 7B model Riverside previously used).
# Predict first: does a 7B model fit on a single A100 80 GB with DDP?
#   (a) Yes — 7B with DDP fits comfortably
#   (b) No — even 7B OOMs with full fp32 Adam states
# Run the cell to check your prediction.

model_params_b_yt = 70  # # CHANGE to 7

p_gb_yt = model_params_b_yt * 1e9 * 2 / 1e9  # bf16 params
g_gb_yt = p_gb_yt  # bf16 grads
o_gb_yt = model_params_b_yt * 1e9 * 8 / 1e9  # fp32 Adam states (m + v)
# full DDP footprint: params + grads + optimizer
total_yt = p_gb_yt + g_gb_yt + o_gb_yt
# does the model fit within a single A100's 80 GB?
fits_yt = total_yt <= 80

print(f"DDP memory for {model_params_b_yt}B model:")
print(f"  Params (bf16):         {p_gb_yt:5.0f} GB")
print(f"  Gradients (bf16):      {g_gb_yt:5.0f} GB")
print(f"  Optimizer (fp32 Adam): {o_gb_yt:5.0f} GB")
print(f"  Total per GPU:         {total_yt:5.0f} GB")
print(
    f"  Fits in A100 80 GB:    {' YES — single-GPU DDP works' if fits_yt else ' NO — OOM even with DDP'}"
)
print()
# 7B is small enough that plain DDP works without any sharding tricks
if fits_yt:
    print(
        "  → The 7B model fits. DDP is the right tool here: simple, no code complexity."
    )
    print("  → The 70B model does NOT — you need FSDP + LoRA + NF4.")

---

## Part 2 — FSDP: Shard Everything Across GPUs

The memory cell in Part 1 gave us the hard number: DDP requires the **full model
on every GPU**. For Riverside's 70B at bf16 with Adam optimizer states, that is
~840 GB per GPU. With 80 GB A100s, DDP is dead on arrival before the first forward pass.

The root problem is DDP's "replicate everything" model. **FSDP's fix:** _shard_ the
parameters, gradients, and optimizer states across GPUs. Each GPU owns only `1/N` of
each tensor. When a layer needs its full weights (forward pass), a fast **all-gather**
reconstructs them temporarily — then immediately frees them. Memory per GPU drops from
`total` to roughly `total / N_GPUs + small_overhead`.

**FSDP (Fully Sharded Data Parallel)** shards parameters, gradients, AND optimizer states across all GPUs:

- Each GPU stores `1/N` of each parameter shard
- Before a layer's forward pass: **all-gather** the full layer weights (temporary)
- After backward: **reduce-scatter** and **free** the gathered weights

Memory per GPU ≈ `total_memory / N_GPUs + communication overhead`

**Riverside scenario: LoRA fine-tuning with NF4 base quantization on 4× A100 80GB**

Full fine-tuning of a 70B model requires 840 GB total — FSDP across 4 GPUs gives 210 GB/GPU, which exceeds 80 GB. In practice, engineers use **LoRA fine-tuning**: the frozen base model is loaded in 4-bit NF4 (≈35 GB total), and only the small adapter parameters (~700M, ≈1% of 70B) need full gradients and optimizer states.

Memory breakdown with LoRA + NF4:

- Base model (70B, NF4 quantized, frozen): `70B × 0.5 bytes ≈ 35 GB` — sharded across 4 GPUs → **~8.75 GB/GPU**
- Adapter params (700M, bf16): `700M × 2 bytes ≈ 1.4 GB`
- Adapter gradients (bf16): `≈ 1.4 GB`
- Adapter optimizer states (fp32 Adam): `700M × 8 bytes ≈ 5.6 GB`
- **Total adapter overhead per GPU: ≈ 8.4 GB** (adapters are small enough to replicate across GPUs)
- **Total per GPU: ~8.75 + 8.4 ≈ 17 GB  fits in 80 GB A100**


### Image Setup: Follow What Is Persistent and What Is Temporary

Before viewing `images/fsdp-vs-ddp-memory.png`, ask two questions of each GPU box:

1. Does it hold a complete persistent model replica, or only its owned shard?
2. When full layer weights appear, how long do they remain allocated?

DDP's repeated full copies explain its simple execution and high memory cost. FSDP's arrows indicate temporary reconstruction: all-gather only the layer needed now, compute, then reshard. Do not interpret the picture as all four GPUs pooling VRAM into one flat 320 GB address space.

![DDP vs FSDP memory: DDP has full amber model copy on every GPU; FSDP has small teal shards with all-gather arrows when a layer is needed](images/fsdp-vs-ddp-memory.png)


### Guided Reading: Gather, Compute, Reshard

In the FSDP side of the image, first identify the small persistent shards. Then follow the arrows that temporarily assemble one layer. The memory saving comes from freeing that full layer before moving on, not from eliminating communication.

```mermaid
stateDiagram-v2
    [*] --> Sharded
    Sharded --> Gathered: all-gather next layer
    Gathered --> Forward: compute activations
    Forward --> Resharded: free full parameters
    Resharded --> GatheredBackward: all-gather for backward
    GatheredBackward --> Reduced: reduce-scatter gradients
    Reduced --> Sharded: optimizer updates owned shard
```

For the notebook's full-fine-tuning convention, one 70B model needs:

- bf16 parameters: $70\text{B} \times 2 = 140$ GB
- bf16 gradients: $70\text{B} \times 2 = 140$ GB
- fp32 Adam states: $70\text{B} \times 8 = 560$ GB
- total persistent training state: $840$ GB

Ideal four-way sharding gives $840 / 4 = 210$ GB per rank **before** activations, temporary gathered parameters, communication buffers, and allocator fragmentation. That is why "FSDP divides memory by four" does not imply "70B full fine-tuning fits in 80 GB." The later NF4 plus LoRA calculation changes what is stored and trained; FSDP alone does not.

###  Predict First: FSDP Memory Savings

DDP per-GPU memory for 70B (full fine-tuning) = **840 GB**. FSDP shards parameters,
gradients, and optimizer states across all 4 GPUs.

How much memory does full fine-tuning of 70B with FSDP need per GPU on 4× A100 80 GB?

**(a)** ~840 GB/GPU — FSDP doesn't reduce per-GPU memory at all  
**(b)** ~210 GB/GPU — sharding reduces it by 4×, but still well over 80 GB  
**(c)** ~20 GB/GPU — sharding brings it comfortably within 80 GB

_Your prediction:_ \_\_\_

↓ Run the next cell to compute the answer.


In [ ]:
#  Part 2: FSDP memory comparison
n_gpus_options = [1, 2, 4, 8]

print("Memory per GPU: DDP vs FSDP for Riverside's 70B model:")
print(f"{'N GPUs':8s}  {'DDP (GB/GPU)':14s}  {'FSDP (GB/GPU)':14s}  {'A100 fits?':10s}")
print("-" * 55)

# Compare DDP (no sharding) vs FSDP (sharded) memory at increasing GPU counts
for n in n_gpus_options:
    ddp_per_gpu = total_per_gpu_ddp  # DDP: full copy on every GPU

    # FSDP: params + grads sharded; optimizer sharded; small all-gather overhead
    fsdp_params_grad = (params_gb + grads_gb) / n  # sharded
    fsdp_optim       = opt_gb / n                  # sharded
    fsdp_allgather   = params_gb / n_gpus_options[-1]  # one layer at a time (tiny)
    # total sharded footprint per GPU
    fsdp_per_gpu = fsdp_params_grad + fsdp_optim + fsdp_allgather

    fits = "\u2713" if fsdp_per_gpu <= a100_vram else "\u2717"
    print(f"  {n:6d}    {ddp_per_gpu:12.0f}    {fsdp_per_gpu:12.0f}    {fits}")

print()
print("Full fine-tuning with FSDP (4 GPUs): still OOM — need LoRA + quantization")
print()

# With LoRA fine-tuning (correct for 4× A100 80GB)
# Base model in NF4 (frozen, inference-only): 70B params × 0.5 bytes ≈ 35 GB total
# Adapter params (trainable, ~1% of 70B): 700M params
adapter_params = 700e6
adapter_bytes_per_param = 2 + 2 + 8  # param + grad + Adam states (fp32)
# total adapter memory overhead (params + grads + optimizer)
adapter_gb = adapter_params * adapter_bytes_per_param / 1e9

# NF4 base model sharded across 4 GPUs
base_nf4_gb = 70e9 * 0.5 / 1e9  # 70B params at ~0.5 bytes each (NF4)
fsdp_4gpu = base_nf4_gb / 4 + adapter_gb  # each GPU holds 1/4 of the frozen base + full adapters
print(f"  LoRA + NF4 FSDP on 4\u00d7 A100 80GB: {fsdp_4gpu:.1f} GB/GPU ({'\u2713 fits!' if fsdp_4gpu <= a100_vram else '\u2717 OOM'})")
print()
print("FSDP code change from DDP:")
print("""
  # DDP:
  model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[rank])

  # FSDP:
  from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
  model = FSDP(model)  # same API; automatic sharding
""")

#### What just happened — and what's missing

The FSDP table confirmed the key tension: **full fine-tuning** of 70B with FSDP across
4 GPUs still needs ~210 GB/GPU — still OOM. Sharding the model is necessary but not
sufficient.

The real solution is a _combination_: FSDP shards the base weights and optimizer states,
LoRA shrinks the number of trainable parameters to ~700M, and NF4 quantization compresses
the frozen base model from ~140 GB to ~35 GB. Together they bring the per-GPU total from
840 GB all the way down to ~17 GB — fitting comfortably on an 80 GB A100.

→ **What's missing:** We have not asked what happens when a single weight matrix is too
wide to fit in memory even momentarily, during the all-gather. FSDP shards across layers,
but tensor parallelism handles the width of each individual layer. That is Part 3.


In [ ]:
# #### Your turn — FSDP memory with a larger LoRA rank
#
# The FSDP analysis above used 1% of 70B as trainable adapter params (~700M).
# # CHANGE `lora_frac_yt` to 0.05 (5% of 70B = 3.5B trainable params).
# Predict first: does a 5% LoRA still fit on 4× A100 80 GB?
#   (a) Yes — even 5% of 70B is small relative to the base model
#   (b) No — 5% adapter overhead overflows the 80 GB budget

lora_frac_yt = 0.01  # # CHANGE to 0.05

# scale trainable adapter params by the chosen LoRA fraction
adapter_params_yt = 70e9 * lora_frac_yt
adapter_bytes_yt = 2 + 2 + 8  # param (bf16) + grad (bf16) + Adam states (fp32)
# total adapter memory overhead at this LoRA fraction
adapter_gb_yt = adapter_params_yt * adapter_bytes_yt / 1e9

base_nf4_gb = 70e9 * 0.5 / 1e9  # frozen NF4 base model (~0.5 bytes/param)
fsdp_yt = base_nf4_gb / 4 + adapter_gb_yt  # NF4 base sharded + full adapters

print(f"FSDP + LoRA ({lora_frac_yt*100:.0f}%) + NF4 on 4× A100 80 GB:")
print(
    f"  Trainable adapter params: {adapter_params_yt/1e9:.1f}B ({lora_frac_yt*100:.0f}% of 70B)"
)
print(
    f"  Adapter overhead:         {adapter_gb_yt:.1f} GB (params + grads + optimizer)"
)
print(f"  NF4 base (sharded / 4):  {base_nf4_gb/4:.1f} GB/GPU")
print(f"  Total per GPU:            {fsdp_yt:.1f} GB")
print(f"  Fits in 80 GB A100:       {' YES' if fsdp_yt <= 80 else ' NO — OOM'}")
print()
# report whether this LoRA fraction still fits within the 80 GB budget
if fsdp_yt <= 80:
    print(
        f"  → {lora_frac_yt*100:.0f}% LoRA fits. Adapter overhead scales linearly with rank."
    )
    print(
        f"  → Key: adapter overhead at 5% = {70e9*0.05*12/1e9:.0f} GB — still manageable."
    )
else:
    print(
        f"  → OOM. At {lora_frac_yt*100:.0f}%, adapter overhead alone is {adapter_gb_yt:.0f} GB."
    )

---

## Part 3 — Tensor Parallelism: Split Individual Layers

Parts 1 and 2 solved the _data parallelism_ and _optimizer memory_ problems. But there is
a subtler limit: in LLaMA-2-70B, the Feed-Forward Network at each layer expands from
hidden_dim 8192 to 4×8192 = 32768. That single weight matrix is shape **(8192, 32768)** —
about 2 GB at bf16. Stack 80 layers and you have 160 GB in FFN weights alone. Even after
FSDP sharding across 4 GPUs, a momentary all-gather of any one layer materializes 2 GB of
floats on a GPU that might already be under pressure.

**Tensor parallelism's fix:** rather than sharding _across time_ (different data, assembled
layer-by-layer), slice the weight matrix _itself_. GPU 0 holds the left columns of W, GPU 1
holds the right columns. Both compute simultaneously and concatenate results. One large
matrix multiply becomes two smaller ones running in parallel — no temporary full
materialization needed.

DDP and FSDP handle memory by sharding parameters **across the time dimension** (different mini-batches or parameter shards). **Tensor parallelism** splits the weight matrix itself — each GPU holds columns/rows of W.

For a linear layer `Y = X @ W` where W is `(D, 4D)`:

- GPU 0 holds columns 0 to 2D-1 of W → computes part of Y
- GPU 1 holds columns 2D to 4D-1 of W → computes part of Y
- Combine: `Y_full = concat([Y_gpu0, Y_gpu1], dim=-1)`

This is how LLaMA-2-70B splits each attention layer across GPUs in the official training recipe.


> **Toy example (4×8 matrix, 2 GPUs):** W is shape (4, 8). GPU 0 holds `W[:, 0:4]` — the left 4 columns. GPU 1 holds `W[:, 4:8]` — the right 4 columns. For input x of shape (batch, 4): GPU 0 computes `x @ W[:, 0:4]` → partial output (batch, 4). GPU 1 computes `x @ W[:, 4:8]` → another (batch, 4). Concatenate both along dim=-1 → (batch, 8). This is bit-identical to computing `x @ W` on one GPU. The communication cost: one `all_gather` per forward pass to reconstruct the full output.


###  Predict First: Column-Parallel Correctness

We split a weight matrix W of shape **(256, 1024)** column-wise across 2 GPUs.
Each GPU holds a **(256, 512)** chunk. After each GPU computes its partial output
`x @ W_chunk`, we concatenate along the last axis to reconstruct the full output.

Is the result identical to computing `x @ W` on a single GPU?

**(a)** Yes — column-parallel split + concatenation is mathematically exact  
**(b)** No — the split introduces floating-point rounding errors that accumulate  
**(c)** No — concatenation is not the right operation; you need addition, not concat

_Your prediction:_ \_\_\_

↓ Run the next cell to verify.


In [ ]:
#  Part 3: Column-parallel linear (tensor parallelism)
torch.manual_seed(42)
D_IN, D_OUT = 256, 1024  # example: D→4D FFN expansion
W_full = torch.randn(D_IN, D_OUT)
b_full = torch.randn(D_OUT)
x_input = torch.randn(8, 32, D_IN)  # (batch, seq, dim)

# Full computation reference
y_full = x_input @ W_full + b_full

# Column-parallel: split W along output dimension (2 "GPUs")
N_GPUS_TP = 2
# size of each GPU's column slice
chunk_size = D_OUT // N_GPUS_TP
W_gpu = [W_full[:, i * chunk_size : (i + 1) * chunk_size] for i in range(N_GPUS_TP)]
b_gpu = [b_full[i * chunk_size : (i + 1) * chunk_size] for i in range(N_GPUS_TP)]

# Each "GPU" computes its partial output
y_partial = [x_input @ W_gpu[i] + b_gpu[i] for i in range(N_GPUS_TP)]

# Gather and concatenate
y_tp = torch.cat(y_partial, dim=-1)

# Verify
match = torch.allclose(y_full, y_tp, atol=1e-5)
print(f"Column-parallel linear (tensor parallelism, {N_GPUS_TP} 'GPUs'):")
print(f"  Full weight shape: {W_full.shape}")
print(f"  Each GPU's weight: {W_gpu[0].shape}")
print(f"  Output matches full computation: {match}")
print()
print(
    f"  Memory savings: each GPU stores {W_gpu[0].numel()*4/1e6:.1f} MB vs {W_full.numel()*4/1e6:.1f} MB total"
)
print(f"  Trade-off: requires one all-gather per forward pass (network bandwidth)")
print()
print("In production (Megatron-LM / llama.cpp):")
print("  - Attention Q,K,V split column-parallel across GPUs")
print("  - Attention output split row-parallel")
print("  - FFN split similarly")
print("  - Communication: one all-reduce per transformer block")

#### What just happened — and what's missing

`torch.allclose` returned `True` — column-parallel linear gave **bit-identical** results
to the full single-GPU computation. Splitting W column-wise and concatenating the partial
outputs is mathematically lossless; no information is lost in the split.

The memory saving is real: each GPU stores one (256 × 512) chunk instead of the full
(256 × 1024) matrix — exactly half the memory per GPU for this layer. In LLaMA-2-70B
with 8-way TP, each GPU holds 1/8 of every attention and FFN weight.

The cost: **one all-gather per forward pass** to reconstruct the full output. In
LLaMA-2-70B, this happens once per transformer block (80 blocks total) — 80 all-gathers
per forward pass, all on the high-bandwidth NVLink interconnect between GPUs.

→ **What's missing:** We have handled layer _width_ (tensor parallel) and parameter
_memory_ (FSDP). Part 4 handles layer _depth_ — what happens when you have 80 layers
that need to be processed across 4 GPUs while keeping them all busy.


In [ ]:
# #### Your turn — Tensor parallelism with 4 GPUs
#
# The demo above used 2 "GPUs". LLaMA-2-70B uses 8-way tensor parallelism.
# # CHANGE `N_GPUS_TP_YT` to 4 and re-run.
# Predict first: does 4-way column-parallel still give identical output?
#   (a) Yes — column-parallel is exact for any N as long as D_OUT % N == 0
#   (b) No — splitting into 4 chunks introduces floating-point error

torch.manual_seed(42)
N_GPUS_TP_YT = 2  # # CHANGE to 4

D_IN_YT, D_OUT_YT = 256, 1024
W_yt = torch.randn(D_IN_YT, D_OUT_YT)
b_yt = torch.randn(D_OUT_YT)
x_yt = torch.randn(8, 32, D_IN_YT)
# reference: full single-GPU computation to compare against
y_full_yt = x_yt @ W_yt + b_yt

# size of each GPU's column slice at the new GPU count
chunk_yt = D_OUT_YT // N_GPUS_TP_YT
# split W and b into N_GPUS_TP_YT column chunks
W_parts = [W_yt[:, i * chunk_yt : (i + 1) * chunk_yt] for i in range(N_GPUS_TP_YT)]
b_parts = [b_yt[i * chunk_yt : (i + 1) * chunk_yt] for i in range(N_GPUS_TP_YT)]
# each simulated GPU computes its partial output independently
y_parts = [x_yt @ W_parts[i] + b_parts[i] for i in range(N_GPUS_TP_YT)]
# concatenate partial outputs back into the full result
y_tp_yt = torch.cat(y_parts, dim=-1)

# confirm the column-parallel result matches the single-GPU reference
match_yt = torch.allclose(y_full_yt, y_tp_yt, atol=1e-5)
print(f"{N_GPUS_TP_YT}-way column-parallel ({D_IN_YT}×{D_OUT_YT} matrix):")
print(f"  Each GPU weight shape: {W_parts[0].shape}")
print(f"  Output matches full computation: {match_yt}")
print(
    f"  Memory per GPU: {W_parts[0].numel()*4/1e6:.2f} MB vs {W_yt.numel()*4/1e6:.2f} MB total"
)
print()
if match_yt:
    print(
        f"  → {N_GPUS_TP_YT}-way column-parallel is exact. Requirement: D_OUT ({D_OUT_YT}) % N ({N_GPUS_TP_YT}) == 0."
    )
    print(
        f"  → LLaMA-2's 8-way TP works: hidden_dim=8192 = 8×1024 , ffn_dim=28672 = 8×3584 "
    )

---

## Part 4 — Pipeline Parallelism: Split Layers Across GPUs

FSDP + tensor parallelism handles the _width_ of each layer. But LLaMA-2-70B has **80
layers**. Even after sharding each layer's weights across GPUs, activations must still
flow through all 80 layers in sequence. With 4 GPUs assigned 20 layers each, the forward
pass becomes a relay race: GPU 0 finishes its 20 layers and passes activations to GPU 1,
which passes to GPU 2, then GPU 3 — three GPUs sit idle while one runs.

**Pipeline parallelism** solves this with **micro-batching**: instead of processing one
large batch end-to-end, split it into micro-batches. While GPU 1 processes micro-batch 1
through its layers, GPU 0 immediately starts micro-batch 2 through its layers. The
pipeline fills up; all GPUs work simultaneously — except for a brief **bubble** at startup
and drain.

**Pipeline parallelism** assigns different **layers** to different GPUs:

- GPU 0: layers 1–20
- GPU 1: layers 21–40
- GPU 2: layers 41–60
- GPU 3: layers 61–80

Each GPU processes one **micro-batch** then passes the activations to the next GPU. While GPU 1 processes micro-batch 1, GPU 0 starts on micro-batch 2. This "fills the pipeline" to reduce idle time.

**Pipeline bubble formula:**

$$\text{bubble} = \frac{N_{\text{stages}} - 1}{N_{\text{stages}} + N_{\text{micro-batches}} - 1}$$

In plain English: the numerator is the number of "warm-up" steps before all GPUs are busy.
The denominator is the total number of steps. As `N_micro-batches` grows, the denominator
grows while the numerator stays fixed — so the bubble fraction shrinks toward zero.
With 4 GPUs and 8 micro-batches: bubble = 3 / (3 + 8 − 1) = 30% idle time.


###  Predict First: Pipeline Bubble Reduction

A 4-GPU pipeline with **4 micro-batches** has a bubble fraction of 30%.
When you double to **8 micro-batches** (keeping 4 pipeline stages), the bubble:

**(a)** Stays at 30% — the bubble is set by GPU count, not micro-batch count  
**(b)** Drops to roughly half (~15%) — more micro-batches fill the pipeline  
**(c)** Drops to roughly 23% — it improves, but not by a factor of two

_Your prediction:_ \_\_\_

↓ Run the next cell to compute the exact values and see the schedule.


In [ ]:
#  Part 4: Pipeline bubble calculation
import matplotlib.patches as mpatches


# Fraction of pipeline time spent idle waiting for the pipeline to fill/drain
def bubble_fraction(n_stages, n_microbatches):
    return (n_stages - 1) / (n_stages + n_microbatches - 1)


n_stages = 4  # 4 GPUs
print("Pipeline efficiency vs. number of micro-batches (4-GPU pipeline):")
print(f"{'Micro-batches':14s}  {'Bubble %':10s}  {'Efficiency':10s}")
# Show how the bubble shrinks as micro-batch count grows
for m in [1, 2, 4, 8, 16, 32]:
    bubble = bubble_fraction(n_stages, m)
    print(f"  {m:12d}  {bubble*100:8.1f}%  {(1-bubble)*100:8.1f}%")

# Visualise a small pipeline
# Draw a Gantt-style chart of which GPU runs which micro-batch over time
fig, ax = plt.subplots(figsize=(12, 5))
n_m = 4  # micro-batches for visualisation
colors = ["steelblue", "coral", "mediumseagreen", "orange"]
# Each stage runs its micro-batches with a diagonal (staggered) start offset
for stage in range(n_stages):
    for mb in range(n_m):
        start = stage + mb  # simple linear pipeline schedule
        ax.barh(
            stage,
            1,
            left=start,
            height=0.7,
            color=colors[mb],
            alpha=0.8,
            edgecolor="white",
        )
        ax.text(
            start + 0.5,
            stage,
            f"M{mb+1}",
            ha="center",
            va="center",
            fontsize=9,
            color="white",
        )

# Shade the trailing bubble region after each stage's micro-batches finish
for stage in range(n_stages):
    bubble_start = stage + n_m
    ax.barh(
        stage,
        n_stages - 1,
        left=bubble_start,
        height=0.7,
        color="lightgray",
        alpha=0.5,
        edgecolor="white",
        hatch="///",
    )

# Label axes and title the pipeline schedule chart
ax.set_yticks(range(n_stages))
ax.set_yticklabels([f"GPU {i}" for i in range(n_stages)])
ax.set_xlabel("Time steps")
ax.set_title(
    "Pipeline schedule (4 GPUs, 4 micro-batches) \u2014 gray = bubble idle time"
)
# Build legend patches: one per micro-batch color, plus the bubble hatch
legend = [
    mpatches.Patch(color=c, label=f"Micro-batch {i+1}") for i, c in enumerate(colors)
]
legend.append(
    mpatches.Patch(color="lightgray", hatch="///", label="Pipeline bubble (idle)")
)
ax.legend(handles=legend, loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

# compute the bubble fraction for the visualized 4-micro-batch case
b4 = bubble_fraction(n_stages, n_m)
print(
    f"\nWith {n_m} micro-batches: {b4*100:.0f}% bubble \u2192 {(1-b4)*100:.0f}% efficiency"
)
print(
    f"With 8 micro-batches: {bubble_fraction(n_stages,8)*100:.0f}% bubble \u2192 {(1-bubble_fraction(n_stages,8))*100:.0f}% efficiency"
)

#### What just happened — and what's missing

The bubble computation showed: with 4 GPUs and 4 micro-batches, **30% of GPU time is
idle**. Doubling micro-batches to 8 cuts it to 23%. You need 32 micro-batches just to
push below 10% idle time. Pipeline parallelism carries a real efficiency cost that does
not disappear — it only shrinks.

This is why pipeline parallelism is typically worth deploying only at 8+ stages, where
the communication savings (not sending full model replicas across node boundaries) outweigh
the bubble overhead. For Riverside's 4-GPU setup, the bubble fraction is too high.

The visualization also shows why the _backward pass_ makes scheduling complex: gradients
must flow in _reverse_ order through the same stages. Implementing this correctly
(without GPUs waiting on each other's backward outputs) is why production pipelines use
dedicated frameworks like Megatron-LM rather than manual orchestration.

→ **What's missing:** We now have all four tools. Part 5 answers which combination
Riverside actually needs — the concrete answer for Monday's CTO meeting.


In [ ]:
# #### Your turn — Pipeline efficiency vs. GPU count
#
# The analysis above used 4 pipeline stages and 4 micro-batches.
# # CHANGE `n_stages_yt` to 8 (what if Riverside had 8 GPUs?).
# Predict first: is an 8-stage pipeline more or less efficient than 4-stage?
#   (a) More efficient — more GPUs means smaller bubble per GPU
#   (b) Less efficient — more stages means a larger bubble for the same micro-batch count


# Same bubble-fraction formula as above, recomputed for the reader's chosen stage count
def bubble_frac(stages, microbatches):
    return (stages - 1) / (stages + microbatches - 1)


n_stages_yt = 4  # # CHANGE to 8
n_microbatches_yt = 8

# bubble fraction for the reader's chosen stage/micro-batch combination
bubble_yt = bubble_frac(n_stages_yt, n_microbatches_yt)
# fraction of time GPUs are actually busy
efficiency_yt = 1 - bubble_yt

print(f"Pipeline: {n_stages_yt} stages, {n_microbatches_yt} micro-batches")
print(f"  Bubble fraction: {bubble_yt*100:.1f}%")
print(f"  Efficiency:      {efficiency_yt*100:.1f}%")
print()
print("Reference — bubble vs. stages and micro-batches:")
print(f"{'Stages':7s}  {'Micro-batches':14s}  {'Bubble':8s}  {'Efficiency':10s}")
# sweep several stage/micro-batch combinations for a reference table
for s, m in [(4, 4), (4, 8), (4, 16), (8, 8), (8, 16), (8, 32)]:
    b = bubble_frac(s, m)
    print(f"  {s:5d}  {m:12d}  {b*100:6.0f}%  {100*(1-b):8.0f}%")
print()
print("  → More stages = larger bubble for the same micro-batch count.")
print("  → Rule of thumb: micro-batches ≥ 4 × n_stages for > 80% efficiency.")

---

## Part 5 — Parallelism Selection: Which Strategy for 70B?

You now have four tools: DDP, FSDP, Tensor Parallel, Pipeline Parallel. The question
is not "which is theoretically best" — it is "which constraints does Riverside's specific
setup actually violate, and which tools fix _exactly_ those violations without adding
unnecessary communication overhead?"

Work through it as a checklist:

1. Does the model fit on one GPU? → No (840 GB > 80 GB). Must shard something.
2. Does FSDP + LoRA + NF4 fit on 4 GPUs? → Yes (~17 GB/GPU). FSDP is sufficient for memory.
3. Does any single layer overflow one GPU even after FSDP sharding? → No (LoRA adapters are tiny). Tensor parallelism is optional.
4. Are there enough GPUs to pipeline stages efficiently? → No (4 GPUs → high bubble fraction). Pipeline parallelism skipped.

Different parallelism axes target different bottlenecks:

| Strategy          | Reduces               | Adds                  | Best for                      |
| ----------------- | --------------------- | --------------------- | ----------------------------- |
| DDP               | — (copies full model) | Gradient all-reduce   | Small models, data bottleneck |
| FSDP              | Parameter memory      | All-gather per layer  | Medium-large models           |
| Tensor parallel   | Per-layer memory      | All-reduce per block  | Very large layers             |
| Pipeline parallel | Cross-GPU memory      | Pipeline bubble       | Very deep models              |
| 3D (all three)    | Maximum memory        | Maximum communication | 70B+ models                   |


### Image Setup: Choose by the Binding Constraint

Before viewing `images/parallelism-strategy-matrix.png`, locate Riverside's point: 70B parameters and 4 GPUs. Read the matrix as a starting region, then test it against the concrete memory arithmetic. Color intensity represents increasing coordination cost, not guaranteed speed.

Use three questions while reading: Does persistent state fit? Can the largest temporary layer fit? Will added stages stay busy? Each "no" motivates a different parallelism axis.

![Parallelism strategy matrix: model size (rows) vs GPU count (columns), colored from simple DDP (light teal) to complex 3D parallel (amber)](images/parallelism-strategy-matrix.png)


### Guided Reading: Stop When the Constraint Is Solved

The matrix moves from replication toward increasingly composed strategies as model size and GPU count grow. It does not say that a more complex cell is always faster. Riverside changes the effective memory problem with NF4 and LoRA, so the simplest sufficient choice is FSDP plus those parameter-efficient techniques.

```mermaid
flowchart TD
    A{Model and training state fit per GPU?}
    A -->|Yes| D[Use DDP]
    A -->|No| B{Largest gathered layer fits?}
    B -->|Yes| F[Use FSDP]
    B -->|No| T[Add tensor parallelism]
    F --> C{Pipeline needed to keep depth manageable?}
    T --> C
    C -->|Yes| P[Add pipeline parallelism]
    C -->|No| S[Stop at simplest sufficient strategy]
```

**Concrete scaling degradation:** suppose one GPU takes 1000 ms per step. Four GPUs reduce ideal compute to 250 ms, but 70 ms of exposed communication gives 320 ms: speedup $1000/320=3.125$ and efficiency $3.125/4=78.1\%$. At eight GPUs, 125 ms of compute plus 95 ms exposed communication gives 220 ms: speedup $4.55$, but efficiency falls to $4.55/8=56.8\%$. More GPUs still help, just less than linearly, because latency, bandwidth, synchronization skew, and smaller work per rank consume a larger fraction of the step.

**Misconceptions to retire:**

- "More GPUs always make training proportionally faster." Communication and stragglers eventually dominate.
- "All-reduce means rank 0 averages and broadcasts." It is a collective; ring implementations distribute the work.
- "Rank means GPU." Rank identifies a process; one-process-per-GPU is a common mapping, not the definition.
- "FSDP makes aggregate cluster VRAM behave like one GPU." Peak gathered layers, activations, and buffers must still fit locally.
- "Overlap makes communication free." It hides only the portion that fits under useful computation; exposed communication remains on the critical path.

###  Predict First: Riverside's Winning Strategy

Going into Part 5 you know:

- DDP: 840 GB/GPU → OOM
- FSDP + LoRA + NF4: ~17 GB/GPU → fits
- Tensor parallel: halves per-layer memory; adds one all-reduce per block
- Pipeline parallel: high bubble at 4 GPUs; typically not worthwhile below 8 stages

For Riverside's 4× A100 80 GB, what is the recommended strategy?

**(a)** Full 3D parallel (FSDP + TP + PP) — use all the tools to be safe  
**(b)** FSDP + LoRA + NF4 — memory is solved; adding TP and PP adds overhead without benefit  
**(c)** FSDP alone (full fine-tuning) — sharding across 4 GPUs should be enough

_Your prediction:_ \_\_\_

↓ Run the next cell to see the strategy comparison with computed numbers.


In [ ]:
#  Part 5: Parallelism selection analysis
print("Parallelism strategy selection for Riverside's 70B on 4\u00d7 A100 80GB:")
print()

# Candidate strategies with their estimated per-GPU memory footprint
strategies = [
    ("DDP only",              total_per_gpu_ddp, "High (full model copy)",  "None"),
    ("FSDP + LoRA + NF4",     fsdp_4gpu,         "Medium (all-gather)",      "None"),
    ("FSDP + Tensor-2way",    fsdp_4gpu / 2,     "Higher (2 comms/block)",  "Megatron style"),
    ("3D (FSDP+TP+PP)",       fsdp_4gpu / 4,     "Highest (3 comms)",       "LLaMA-2-70B recipe"),
]

print(f"{'Strategy':25s}  {'GB/GPU':8s}  {'Fits 80GB?':10s}  {'Communication':20s}  {'Used in':20s}")
print("-" * 95)
# Check each strategy against the 80 GB A100 budget
for name, mem, comm, ref in strategies:
    fits = "\u2713" if mem <= a100_vram else "\u2717"
    print(f"  {name:23s}  {mem:6.0f}    {fits:8s}   {comm:20s}  {ref}")

print()
print("RECOMMENDATION for Riverside 70B on 4\u00d7 A100 80GB:")
print("  Use FSDP (reduces model copy) + Tensor Parallelism across 2 pairs of GPUs")
print(f"  Estimated memory per GPU: ~{fsdp_4gpu/2:.0f} GB ({'\u2713 fits' if fsdp_4gpu/2 <= 80 else '\u2717 OOM'})")
print()
print("Code change from single-GPU:")
print("  Single GPU: model = MyModel()")
print("  FSDP:       model = FSDP(MyModel(), device_id=rank)")
print("  ~5 lines changed in the training script")


#### What just happened — and what's missing

The strategy comparison confirmed the analysis: for Riverside's 4× A100 80 GB + 70B +
LoRA fine-tuning, **FSDP + LoRA + NF4 is sufficient** (~17 GB/GPU). Adding tensor
parallelism would reduce memory further but introduces two all-reduce operations per
transformer block — an overhead not justified when you are already comfortably within 80 GB.

The general principle: **match the parallelism strategy to the binding constraint, then stop**.
Over-engineering the parallelism strategy (using 3D parallel when FSDP alone works) adds
code complexity and communication overhead without benefit.

- Memory bound → FSDP first
- Single layer too wide to all-gather temporarily → add Tensor Parallel
- Too many layers, too few GPUs busy → add Pipeline Parallel
- All three constraints → that is the 3D parallel regime used at 70B+ and 1000+ GPUs

→ **What's missing:** We derived this from first principles. Part 6 asks: does this
match what Meta actually shipped for LLaMA-2-70B at 2048 GPU scale?


---

## Part 6 — Toy → Real: LLaMA-2-70B Training Recipe

The actual LLaMA-2-70B training used exactly the combination we derived:


### Toy → Production Bridge

Before reading the LLaMA-2-70B config, here is how the toy parameters used in this
notebook map to the production numbers you are about to see:

| Parameter                       | This notebook (toy/simulated) | LLaMA-2-70B (Meta, 2023)   |
| ------------------------------- | ----------------------------- | -------------------------- |
| Model params                    | 70B (computed analytically)   | 70B (real weights)         |
| ToyTransformer hidden dim       | 64                            | 8192                       |
| ToyTransformer layers           | 3                             | 80                         |
| TP weight matrix shape (Part 3) | (256, 1024)                   | (8192, 28672) per FFN proj |
| Tensor parallel degree (Part 3) | 2-way (demo)                  | 8-way                      |
| Pipeline stages (Part 4)        | 4 (demo)                      | 16                         |
| Data parallel degree            | —                             | 16                         |
| Total GPUs                      | 4 (Riverside)                 | 2,048 × A100 80 GB         |
| Global batch size               | —                             | 4M tokens                  |
| Micro-batches (Part 4)          | 4–8 (demo)                    | 4 per pipeline stage       |

The gradient math, the bubble formula, and the memory calculations you derived are
**identical** to the mechanisms running at Meta's scale — just with different numbers.


In [ ]:
#  Part 6: LLaMA-2-70B training configuration
# Meta's actual published LLaMA-2-70B training configuration
llama_config = {
    "params": "70B",
    "n_layers": 80,
    "hidden_dim": 8192,
    "n_heads": 64,
    "n_kv_heads": 8,  # GQA
    "training_gpus": "2048\u00d7 A100 80GB",
    "data_parallelism": "16-way DDP equivalent",
    "tensor_parallelism": "8-way column/row parallel",
    "pipeline_stages": "16-stage pipeline",
    "micro_batch": 4,
    "global_batch": "4M tokens",
    "bf16": True,
    "gradient_checkpointing": True,
    "optimizer": "AdamW (ZeRO sharding for optimizer states)",
}

print("LLaMA-2-70B training configuration (Meta AI, 2023):")
# Print every config field for comparison against this notebook's derivation
for k, v in llama_config.items():
    print(f"  {k:25s}: {v}")

print()
print("Mapping to this notebook's concepts:")
print("  - Tensor parallelism (8-way) \u2192 Part 3: column/row parallel attention")
print("  - Pipeline stages (16) \u2192 Part 4: layers split across GPUs")
print("  - ZeRO optimizer sharding \u2192 Part 2: FSDP-like optimizer memory reduction")
print("  - Gradient checkpointing \u2192 Ch2 Part 4: activation recomputation")
print("  - bf16 training \u2192 Ch2 Part 2: precision formats")
print()
print(
    f"Riverside's 70B fine-tuning on 4\u00d7 A100: a scaled-down version of the same recipe."
)
print(f"  Use FSDP (equivalent to ZeRO-3) + bf16 + gradient checkpointing")
print(f"  Pipeline stages: not needed at 4 GPUs (too few for efficient pipelining)")

#### What just happened — and what's missing

The LLaMA-2-70B config confirmed the derivation: Meta used **8-way tensor parallelism**,
**16 pipeline stages**, and **16-way data parallelism** at 2048 GPUs — exactly the 3D
parallel combination that our checklist predicted would be needed when all three
constraints (layer width, layer depth, and data throughput) are simultaneously active
at thousand-GPU scale.

For Riverside at 4 GPUs, the same checklist said: FSDP + LoRA + NF4 is sufficient.
The full 3D recipe is not needed until you hit the regime where even FSDP + LoRA
cannot bring per-GPU memory within budget.

**The notebook is complete.** Every strategy was derived from memory constraints and
bubble arithmetic — not memorized from a documentation page.


---

## Summary: Riverside's Decision

The Platform Engineer's question has a concrete answer. We derived it step by step:

| Part | Finding | Riverside implication |
|------|---------|----------------------|
| 1 — DDP | Each GPU needs 840 GB (params + grads + optimizer) |  OOM: 70B won't fit on any single A100 with DDP |
| 2 — FSDP | LoRA + NF4 quantization brings it to ~17 GB/GPU |  FSDP + LoRA + NF4 fits on 4× A100 80GB |
| 3 — TP | Column-parallel splits layers; verified numerically | Optional for 70B at 4 GPUs; adds communication overhead |
| 4 — PP | Bubble fraction drops as micro-batches increase | Not worthwhile at only 4 GPUs |
| 5 — Selection | FSDP + LoRA + NF4 is sufficient for 4× A100 | ~13 lines of code changed from single-GPU baseline |
| 6 — LLaMA-2 | Meta used TP+PP+DP at 2048 GPUs | Same concepts, different scale |

---

### Key insights to keep

- **DDP all-reduces, not sums** — after backward, every GPU holds the _averaged_ gradient; weights stay bit-identical across all GPUs.
- **DDP's hidden cost is replication** — it scales throughput, not memory; every GPU carries the full model, always.
- **FSDP alone does not save 70B on 4 GPUs** — full fine-tuning still needs ~210 GB/GPU; LoRA + NF4 is what gets you to ~17 GB/GPU.
- **Column-parallel linear is lossless** — splitting W column-wise and concatenating outputs is mathematically identical to the single-GPU computation.
- **Pipeline bubbles never reach zero** — they shrink with more micro-batches but never disappear; need ≥ 4 × n_stages micro-batches for > 80% efficiency.
- **Match the strategy to the binding constraint** — memory OOM → FSDP first; single layer too wide → add TP; too many layers to fit → add PP.
- **~13 lines of code** separate a single-GPU training loop from a distributed FSDP loop — the hard part is knowing _which_ wrapper to reach for, not writing it.


In [ ]:
#  Closing Decision
print("=" * 60)
print("  CLOSING DECISION \u2014 Riverside 70B Parallelism Strategy")
print("=" * 60)
print()
print("  Hardware: 4\u00d7 A100 80GB")
print(f"  Model:    LLaMA-3-70B (bf16 = {params_gb:.0f} GB params)")
print()
print(f"  DDP alone:  {total_per_gpu_ddp:.0f} GB/GPU \u2192 \u2717 OOM")
print(f"  FSDP + LoRA + NF4: {fsdp_4gpu:.0f} GB/GPU \u2192 {'\u2713 fits' if fsdp_4gpu <= a100_vram else '\u2717 OOM, need TP too'}")
print()
print("  RECOMMENDATION: FSDP + bf16 + gradient checkpointing")
print("    from torch.distributed.fsdp import FullyShardedDataParallel as FSDP")
print("    model = FSDP(model, mixed_precision=MixedPrecision(param_dtype=torch.bfloat16))")
print()
print("  Code change vs. single-GPU baseline:")
print("    Add FSDP wrapper: ~5 lines")
print("    Add bf16 mixed precision config: ~3 lines")
print("    Checkpoint every N layers: ~5 lines")
print("    Total: ~13 lines changed, rest of training loop unchanged")
print()
print("  Estimated throughput (reference, 4\u00d7 A100):")
print("    ~200 tokens/s training (LLaMA-3-70B, batch=2, seq=2048)")
print("    Riverside's 7-novel corpus (~2M tokens): ~3 hours per epoch")

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated

- DDP gradient all-reduce — simulated; proved gradients are averaged (not summed)
- DDP vs FSDP memory analysis — computed per-GPU memory for 70B at each strategy
- Tensor parallelism — column-parallel linear implemented and verified
- Pipeline parallelism — bubble fraction computed; schedule visualised
- Parallelism selection — strategy matrix for Riverside 70B case
- LLaMA-2-70B training recipe — mapped to the concepts built in this notebook

### Tier 2 — Explained but Not Fully Implemented

- **FSDP full end-to-end** — the FSDP wrapper call is shown but a full distributed training loop requires `torch.distributed.init_process_group` which needs multiple GPU processes

### Tier 3 — Named but Out of Scope

- **Megatron-LM** — NVIDIA's framework for 3D parallelism; production choice for 70B+ training; uses the same TP/PP/DP concepts built here
- **DeepSpeed ZeRO-3** — Microsoft's optimizer sharding; similar to FSDP; different API
- **Sequence parallelism** — shard the sequence axis across GPUs; reduces activation memory for very long contexts


---

## When to Use What

| Model size | GPUs     | Strategy                            | Why                                          |
| ---------- | -------- | ----------------------------------- | -------------------------------------------- |
| ≤ 3B       | 1 GPU    | Single GPU + gradient checkpointing | Fits with FSDP tricks                        |
| 3–13B      | 1–4 GPUs | FSDP + bf16                         | Shards optimizer states; minimal code change |
| 13–70B     | 4–8 GPUs | FSDP + bf16 + optional TP           | FSDP may suffice; add TP for marginal cases  |
| 70B+       | 8+ GPUs  | 3D (DP + TP + PP)                   | Megatron-LM or DeepSpeed                     |

→ **Next:** `learning/ai-infrastructure/06-quantization/` — after training, the 70B model needs to be compressed for deployment. This is where int4 quantization and GGUF come in.


---

## Production and Cloud Distributed Training

The notebook's Riverside recommendation, **FSDP + LoRA + NF4 on 4 A100 80 GB GPUs**, is only the training algorithm. A production run also needs an explicit control plane:

1. **Launcher and job spec:** Generate a versioned `torchrun` command and an immutable scheduler manifest (Kubernetes, Slurm, or a managed training service). Pin the image digest, code revision, dataset version, secrets by reference, GPU type/count, CPU, memory, and ephemeral storage. Never hide material defaults in a shell script.
2. **Rendezvous:** Use a stable run ID and a fault-tolerant rendezvous backend shared by every node. Configure minimum/maximum nodes, endpoint, timeout, and restart count; do not use a laptop hostname or an ephemeral local port for a multi-node cloud run.
3. **Topology:** Record node count, GPUs per node, world size, and the DP/TP/PP degrees. Require `world_size = dp × tp × pp`. For Riverside, TP and PP remain 1, so all four ranks form the FSDP data-parallel group.
4. **Checkpointing:** Save model/adapters, optimizer, scheduler, scaler, RNG state, sampler position, and progress at a globally consistent step. FSDP checkpoints must declare full versus sharded state-dict format. Write payloads first and atomically publish a small metadata pointer only after every required shard is durable.
5. **Fault tolerance and resume:** Treat preemption and rank loss as expected. Bound retries, emit heartbeats, keep checkpoints on remote durable storage, and resume only when the code, model, data, topology, and checkpoint schema are compatible. Restore the exact global step and data cursor instead of replaying samples silently.
6. **Gradient accumulation:** Derive effective batch size explicitly: `global_batch = micro_batch_per_gpu × data_parallel_degree × accumulation_steps`. Synchronize gradients only on the final micro-step (`no_sync()` for earlier DDP/FSDP micro-steps), and checkpoint only after an optimizer step.
7. **Deterministic seeds:** Store a base seed and derive rank-specific data/worker seeds. Capture Python, NumPy, CPU CUDA, and per-device CUDA RNG states. Determinism can reduce throughput, so record the selected deterministic-algorithm policy with the run.
8. **Observability:** Log rank-zero metrics plus per-rank health, throughput, tokens/sec, step time, data wait, communication time, GPU utilization/memory, gradient norm, loss scale, checkpoint latency, retries, and rendezvous events. Tag logs and traces with run ID, rank, node, code revision, and checkpoint ID.
9. **Quotas and cost:** Validate regional GPU quota and scheduler capacity before submission. Set a maximum node count, runtime deadline, retry budget, storage lifecycle, and cost estimate; alert on idle GPUs, low scaling efficiency, and repeated restarts.
10. **Artifact promotion:** Training output is not automatically a release. Promote an immutable adapter/model artifact only after completeness, checksum, evaluation, safety, lineage, and reproducibility gates pass. Keep the source checkpoint and evaluation report linked to the promoted registry version.

The following cells are **planning and metadata examples only**. Every `RUN_PRODUCTION_*` switch defaults to `False`; the notebook does not launch processes, submit jobs, contact cloud services, or write checkpoints unless a reader deliberately changes a guard.

In [ ]:
# Production configuration and environment validation (planning only)
from dataclasses import asdict, dataclass
from typing import Any
import json
import os
import platform

RUN_PRODUCTION_ENV_VALIDATION = False
RUN_PRODUCTION_LAUNCH = False  # This notebook never launches a distributed job.


@dataclass(frozen=True)
class ProductionTrainingConfig:
    run_id: str
    image: str
    code_revision: str
    dataset_version: str
    model_id: str
    strategy: str
    nodes: int
    gpus_per_node: int
    data_parallel_degree: int
    tensor_parallel_degree: int
    pipeline_parallel_degree: int
    micro_batch_per_gpu: int
    gradient_accumulation_steps: int
    rendezvous_backend: str
    rendezvous_endpoint: str
    rendezvous_timeout_seconds: int
    max_restarts: int
    checkpoint_uri: str
    checkpoint_format: str
    base_seed: int
    deterministic_algorithms: bool
    max_runtime_hours: float
    gpu_hour_budget: float
    regional_gpu_quota: int

    @property
    def world_size(self) -> int:
        return self.nodes * self.gpus_per_node

    @property
    def effective_global_batch(self) -> int:
        return (
            self.micro_batch_per_gpu
            * self.data_parallel_degree
            * self.gradient_accumulation_steps
        )


production_config = ProductionTrainingConfig(
    run_id="riverside-70b-lora-v1",
    image="registry.example/riverside-trainer@sha256:" + "0" * 64,
    code_revision="0123456789abcdef0123456789abcdef01234567",
    dataset_version="novels-v3-sha256-8f2c1d",
    model_id="llama-70b-base-v1",
    strategy="fsdp_lora_nf4",
    nodes=1,
    gpus_per_node=4,
    data_parallel_degree=4,
    tensor_parallel_degree=1,
    pipeline_parallel_degree=1,
    micro_batch_per_gpu=1,
    gradient_accumulation_steps=8,
    rendezvous_backend="c10d",
    rendezvous_endpoint="rendezvous.riverside.internal:29400",
    rendezvous_timeout_seconds=900,
    max_restarts=3,
    checkpoint_uri="s3://riverside-training/checkpoints/riverside-70b-lora-v1",
    checkpoint_format="fsdp_sharded_state_dict_v1",
    base_seed=20250308,
    deterministic_algorithms=False,
    max_runtime_hours=6.0,
    gpu_hour_budget=24.0,
    regional_gpu_quota=4,
)


def validate_production_config(config: ProductionTrainingConfig) -> dict[str, Any]:
    errors: list[str] = []
    warnings: list[str] = []
    topology_product = (
        config.data_parallel_degree
        * config.tensor_parallel_degree
        * config.pipeline_parallel_degree
    )

    if config.world_size != topology_product:
        errors.append(
            "world_size must equal data_parallel_degree × tensor_parallel_degree × "
            "pipeline_parallel_degree"
        )
    if config.strategy == "fsdp_lora_nf4" and (
        config.tensor_parallel_degree != 1 or config.pipeline_parallel_degree != 1
    ):
        warnings.append("Riverside's selected FSDP-only topology expects TP=1 and PP=1")
    if config.effective_global_batch <= 0:
        errors.append("effective global batch must be positive")
    if not config.rendezvous_endpoint or ":" not in config.rendezvous_endpoint:
        errors.append("rendezvous_endpoint must be a shared host:port")
    if config.rendezvous_backend not in {"c10d", "etcd-v2"}:
        errors.append("unsupported elastic rendezvous backend")
    if config.rendezvous_timeout_seconds < 60:
        errors.append("rendezvous timeout is too short for cloud node startup")
    if config.max_restarts < 0:
        errors.append("max_restarts cannot be negative")
    if config.regional_gpu_quota < config.world_size:
        errors.append("regional GPU quota is below requested world size")
    estimated_gpu_hours = config.world_size * config.max_runtime_hours
    if estimated_gpu_hours > config.gpu_hour_budget:
        errors.append("maximum runtime exceeds the configured GPU-hour budget")
    if "@sha256:" not in config.image:
        errors.append("container image must be pinned by digest")
    if len(config.code_revision) != 40:
        errors.append("code_revision must be an immutable 40-character Git SHA")
    if not config.checkpoint_uri.startswith(("s3://", "gs://", "az://")):
        errors.append("production checkpoints must use durable remote object storage")
    if config.checkpoint_format != "fsdp_sharded_state_dict_v1":
        errors.append("checkpoint format must match the Riverside FSDP save/load path")

    environment = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "cuda_visible_devices": os.getenv("CUDA_VISIBLE_DEVICES", "not set"),
        "scheduler_job_id": os.getenv("JOB_ID", "not set"),
    }
    return {
        "valid": not errors,
        "errors": errors,
        "warnings": warnings,
        "world_size": config.world_size,
        "effective_global_batch": config.effective_global_batch,
        "estimated_gpu_hours": estimated_gpu_hours,
        "environment": environment,
    }


validation_report = validate_production_config(production_config)
if RUN_PRODUCTION_ENV_VALIDATION:
    if not validation_report["valid"]:
        raise ValueError(json.dumps(validation_report, indent=2))
    print(json.dumps({"config": asdict(production_config), "validation": validation_report}, indent=2))
else:
    print("Production validation is disabled; set RUN_PRODUCTION_ENV_VALIDATION=True to inspect locally.")
    print(
        f"Planned topology: world_size={production_config.world_size}, "
        f"global_batch={production_config.effective_global_batch}, "
        f"validation_ready={validation_report['valid']}"
    )

In [ ]:
# Build an auditable launch plan and scheduler manifest without submitting it
RUN_PRODUCTION_MANIFEST_WRITE = False


def build_launch_plan(config: ProductionTrainingConfig) -> dict[str, Any]:
    if RUN_PRODUCTION_LAUNCH:
        raise RuntimeError("Process and distributed-job launch is intentionally disabled in this notebook")

    torchrun_argv = [
        "torchrun",
        f"--nnodes={config.nodes}:{config.nodes}",
        f"--nproc-per-node={config.gpus_per_node}",
        f"--rdzv-id={config.run_id}",
        f"--rdzv-backend={config.rendezvous_backend}",
        f"--rdzv-endpoint={config.rendezvous_endpoint}",
        f"--max-restarts={config.max_restarts}",
        "train_fsdp_lora.py",
        f"--model-id={config.model_id}",
        f"--dataset-version={config.dataset_version}",
        f"--micro-batch={config.micro_batch_per_gpu}",
        f"--gradient-accumulation={config.gradient_accumulation_steps}",
        f"--checkpoint-uri={config.checkpoint_uri}",
        f"--seed={config.base_seed}",
    ]

    manifest = {
        "apiVersion": "training.example/v1",
        "kind": "DistributedTrainingJob",
        "metadata": {
            "name": config.run_id,
            "labels": {
                "strategy": config.strategy,
                "code-revision": config.code_revision[:12],
                "dataset-version": config.dataset_version,
            },
        },
        "spec": {
            "suspend": True,
            "image": config.image,
            "command": torchrun_argv,
            "replicas": config.nodes,
            "processesPerReplica": config.gpus_per_node,
            "topology": {
                "worldSize": config.world_size,
                "dataParallel": config.data_parallel_degree,
                "tensorParallel": config.tensor_parallel_degree,
                "pipelineParallel": config.pipeline_parallel_degree,
            },
            "resourcesPerReplica": {
                "gpu": config.gpus_per_node,
                "gpuType": "A100-80GB",
                "cpu": 48,
                "memoryGiB": 384,
                "ephemeralStorageGiB": 500,
            },
            "rendezvous": {
                "backend": config.rendezvous_backend,
                "endpoint": config.rendezvous_endpoint,
                "timeoutSeconds": config.rendezvous_timeout_seconds,
            },
            "faultTolerance": {
                "maxRestarts": config.max_restarts,
                "checkpointUri": config.checkpoint_uri,
                "terminationGracePeriodSeconds": 300,
            },
            "observability": {
                "rankZeroMetrics": ["loss", "tokens_per_second", "gradient_norm"],
                "perRankMetrics": [
                    "step_time_seconds",
                    "data_wait_seconds",
                    "gpu_utilization",
                    "gpu_memory_bytes",
                    "checkpoint_latency_seconds",
                ],
                "logFields": ["run_id", "rank", "node", "code_revision", "checkpoint_id"],
            },
            "guardrails": {
                "activeDeadlineSeconds": int(config.max_runtime_hours * 3600),
                "requiredRegionalGpuQuota": config.world_size,
                "gpuHourBudget": config.gpu_hour_budget,
                "artifactPromotion": "manual_after_evaluation",
            },
        },
    }
    return {
        "launcher": torchrun_argv,
        "manifest": manifest,
        "estimated_gpu_hours": config.world_size * config.max_runtime_hours,
    }


launch_plan = build_launch_plan(production_config)
print("Launch plan generated as inert data; manifest spec.suspend is True.")
print("Launcher preview:", " ".join(launch_plan["launcher"]))
print(json.dumps(launch_plan["manifest"], indent=2))

if RUN_PRODUCTION_MANIFEST_WRITE:
    raise RuntimeError(
        "Manifest writes are disabled in this notebook; export the validated plan in deployment tooling"
    )

In [ ]:
# Atomic checkpoint metadata, deterministic resume, and artifact promotion gates
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import tempfile

RUN_PRODUCTION_CHECKPOINT_IO = False
RUN_PRODUCTION_RESUME = False
RUN_PRODUCTION_ARTIFACT_PROMOTION = False
CHECKPOINT_SCHEMA_VERSION = 1


def derive_rank_seed(base_seed: int, rank: int, worker_id: int = 0) -> int:
    seed_material = f"{base_seed}:{rank}:{worker_id}".encode("utf-8")
    return int.from_bytes(hashlib.sha256(seed_material).digest()[:4], "big")


def build_checkpoint_metadata(
    config: ProductionTrainingConfig,
    global_step: int,
    epoch: int,
    samples_consumed: int,
    shard_checksums: dict[str, str],
) -> dict[str, Any]:
    if global_step < 0 or samples_consumed < 0:
        raise ValueError("checkpoint progress cannot be negative")
    if global_step % 1 != 0:
        raise ValueError("checkpoint must be published after a complete optimizer step")
    if len(shard_checksums) != config.world_size:
        raise ValueError("a sharded FSDP checkpoint requires one durable shard per rank")
    if any(len(digest) != 64 for digest in shard_checksums.values()):
        raise ValueError("every checkpoint shard must have a SHA-256 digest")

    checkpoint_id = f"step-{global_step:012d}"
    return {
        "schema_version": CHECKPOINT_SCHEMA_VERSION,
        "checkpoint_id": checkpoint_id,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "run_id": config.run_id,
        "global_step": global_step,
        "epoch": epoch,
        "samples_consumed": samples_consumed,
        "optimizer_step_complete": True,
        "checkpoint_format": config.checkpoint_format,
        "world_size": config.world_size,
        "topology": {
            "dp": config.data_parallel_degree,
            "tp": config.tensor_parallel_degree,
            "pp": config.pipeline_parallel_degree,
        },
        "lineage": {
            "image": config.image,
            "code_revision": config.code_revision,
            "dataset_version": config.dataset_version,
            "model_id": config.model_id,
        },
        "rng": {
            "base_seed": config.base_seed,
            "rank_seeds": {
                str(rank): derive_rank_seed(config.base_seed, rank)
                for rank in range(config.world_size)
            },
            "capture_required": [
                "python_random",
                "numpy",
                "torch_cpu",
                "torch_cuda_per_device",
                "dataloader_sampler",
            ],
            "deterministic_algorithms": config.deterministic_algorithms,
        },
        "required_state": [
            "lora_adapter_shards",
            "optimizer_shards",
            "scheduler",
            "gradient_scaler",
            "rng_state",
            "sampler_cursor",
        ],
        "shards": shard_checksums,
        "complete": True,
    }


def atomic_write_json(path: Path, payload: dict[str, Any]) -> None:
    """Publish metadata only after checkpoint payloads and checksums are durable."""
    path.parent.mkdir(parents=True, exist_ok=True)
    serialized = json.dumps(payload, indent=2, sort_keys=True) + "\n"
    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        dir=path.parent,
        prefix=f".{path.name}.",
        suffix=".tmp",
        delete=False,
    ) as temporary_file:
        temporary_file.write(serialized)
        temporary_file.flush()
        os.fsync(temporary_file.fileno())
        temporary_path = Path(temporary_file.name)
    os.replace(temporary_path, path)


def validate_resume(
    config: ProductionTrainingConfig,
    metadata: dict[str, Any],
) -> list[str]:
    incompatibilities: list[str] = []
    expected_lineage = {
        "image": config.image,
        "code_revision": config.code_revision,
        "dataset_version": config.dataset_version,
        "model_id": config.model_id,
    }
    expected_topology = {
        "dp": config.data_parallel_degree,
        "tp": config.tensor_parallel_degree,
        "pp": config.pipeline_parallel_degree,
    }

    if metadata.get("schema_version") != CHECKPOINT_SCHEMA_VERSION:
        incompatibilities.append("checkpoint schema version differs")
    if not metadata.get("complete") or not metadata.get("optimizer_step_complete"):
        incompatibilities.append("checkpoint was not atomically completed after an optimizer step")
    if metadata.get("checkpoint_format") != config.checkpoint_format:
        incompatibilities.append("FSDP checkpoint format differs")
    if metadata.get("world_size") != config.world_size:
        incompatibilities.append("world size differs; explicit checkpoint resharding is required")
    if metadata.get("topology") != expected_topology:
        incompatibilities.append("DP/TP/PP topology differs")
    if metadata.get("lineage") != expected_lineage:
        incompatibilities.append("image, code, data, or base-model lineage differs")
    if len(metadata.get("shards", {})) != config.world_size:
        incompatibilities.append("checkpoint shard set is incomplete")
    return incompatibilities


def evaluate_artifact_promotion(
    metadata: dict[str, Any],
    evaluation: dict[str, bool],
) -> dict[str, Any]:
    required_gates = {
        "checkpoint_complete": bool(metadata.get("complete")),
        "all_shards_present": len(metadata.get("shards", {})) == metadata.get("world_size"),
        "checksums_verified": evaluation.get("checksums_verified", False),
        "quality_passed": evaluation.get("quality_passed", False),
        "safety_passed": evaluation.get("safety_passed", False),
        "lineage_recorded": bool(metadata.get("lineage")),
        "reproducibility_report_attached": evaluation.get(
            "reproducibility_report_attached", False
        ),
    }
    failed_gates = [name for name, passed in required_gates.items() if not passed]
    return {
        "eligible": not failed_gates,
        "failed_gates": failed_gates,
        "source_checkpoint_id": metadata.get("checkpoint_id"),
        "promotion_mode": "immutable_registry_version",
    }


example_shards = {
    f"rank-{rank:05d}.distcp": hashlib.sha256(f"example-shard-{rank}".encode()).hexdigest()
    for rank in range(production_config.world_size)
}
checkpoint_metadata = build_checkpoint_metadata(
    production_config,
    global_step=1200,
    epoch=2,
    samples_consumed=production_config.effective_global_batch * 1200,
    shard_checksums=example_shards,
)
resume_issues = validate_resume(production_config, checkpoint_metadata)
promotion_report = evaluate_artifact_promotion(
    checkpoint_metadata,
    {
        "checksums_verified": True,
        "quality_passed": True,
        "safety_passed": True,
        "reproducibility_report_attached": True,
    },
)

if RUN_PRODUCTION_CHECKPOINT_IO:
    atomic_write_json(Path("production-checkpoints/latest.json"), checkpoint_metadata)
if RUN_PRODUCTION_RESUME:
    if resume_issues:
        raise RuntimeError("Resume rejected: " + "; ".join(resume_issues))
    print("Resume metadata is compatible; restore state in the external training entry point.")
if RUN_PRODUCTION_ARTIFACT_PROMOTION:
    if not promotion_report["eligible"]:
        raise RuntimeError("Artifact promotion blocked: " + ", ".join(promotion_report["failed_gates"]))
    print("Promotion gates passed; registry publication remains an external deployment action.")

print(
    f"Checkpoint metadata prepared for {checkpoint_metadata['checkpoint_id']}; "
    f"resume_issues={len(resume_issues)}, promotion_eligible={promotion_report['eligible']}."
)
print("No checkpoint files, processes, jobs, or registry artifacts were created.")